# Create DRAGON Dataset for DeReC Fine-tuning

DRAGON has no explicit "grounded/ungrounded" labels, so we create a synthetic evaluation dataset:
- **Grounded**: LLM answers using ground truth documents as context (label=1)
- **Ungrounded**: LLM answers WITHOUT context (label=0)

We iterate over all dataset versions (1.0.0 → 1.15.0) to collect as many unique questions as possible.
Ground truth document IDs come from the private QA dataset (`text_ids`), so no retriever is needed.
Published to HuggingFace for DeReC fine-tuning.

## 1. Setup

In [1]:
# ! rm -r rag_fact_checking
!git clone -b feature/evaluate-fact-checking https://github.com/BigMak1/rag_fact_checking.git
# !git -C rag_fact_checking pull

fatal: destination path 'rag_fact_checking' already exists and is not an empty directory.


In [2]:
# import os
# os.makedirs("/kaggle/working/my_packages", exist_ok=True)
# !pip download -r rag_fact_checking/DRAGON/requirements.txt -d /kaggle/working/my_packages
# !pip install --no-index --find-links /kaggle/input/my-custom-packages -r rag_fact_checking/DRAGON/requirements.txt

In [3]:
%%time
%%capture
# !pip install -q -r rag_fact_checking/DRAGON/requirements.txt
!uv pip install --system -r rag_fact_checking/DRAGON/requirements.txt

CPU times: user 2.31 ms, sys: 7.98 ms, total: 10.3 ms
Wall time: 268 ms


In [4]:
import ast
import json
import random
import os
from collections import defaultdict
import gc

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from datasets import load_dataset, Dataset, DatasetDict
from langchain_community.llms import VLLM
import numpy as np
import torch
from transformers import AutoTokenizer
from tqdm.auto import tqdm

from rag_fact_checking.DRAGON.rag_bench import results
from rag_fact_checking.DRAGON.rag_bench.helper import get_ds_versions, sort_versions
from rag_fact_checking.DRAGON.rag_bench.constants import HIST_TEXTS_REPO_ID, HIST_QUESTIONS_REPO_ID

In [5]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

In [ ]:
# ── Constants ──
HIST_PRIVATE_QA_REPO_ID: str = "ai-forever/hist-rag-bench-private-qa"
HIST_PRIVATE_TEXTS_REPO_ID: str = "ai-forever/hist-rag-bench-private-texts"
RANDOM_SEED: int = 42
LLM_NAME: str = "bond005/meno-tiny-0.1"

HF_TOKEN: str = user_secrets.get_secret("HF_TOKEN")
HF_DATASET_REPO: str = "Makson4ic/dragon-derec-dataset"

# Version range to process
MIN_VERSION: str = "1.0.0"
MAX_VERSION: str = "1.15.0"

# ── Prompts ──
LLM_PROMPT: str = """Проанализируйте заданный контекст и ответьте на вопрос пользователя на основе сведений, предоставленных в этом контексте.
Не давайте никаких объяснений и пояснений к своему ответу. Не пишите ничего лишнего. Не извиняйтесь, не стройте диалог. Выдавайте только ответ и ничего больше.
Отвечайте на русском языке.
Если в заданном контексте нет информации для ответа на вопрос пользователя, то ничего не придумывайте и просто откажитесь отвечать.
"""

LLM_PROMPT_NO_CONTEXT: str = """Ответьте на вопрос пользователя.
Не давайте никаких объяснений. Выдавайте только ответ и ничего больше.
Отвечайте на русском языке.
"""

In [7]:
# ── Helpers ──
def _build_question_index(questions_ds):
    """Map str(question_id) -> {"question": ...}"""
    idx = {}
    for item in questions_ds["train"]:
        idx[str(item["id"])] = {"question": item["question"]}
    return idx


def _build_text_index(texts_ds):
    """Map doc_id -> text content (public texts)"""
    idx = {}
    for item in texts_ds["train"]:
        idx[item["id"]] = item["text"]
    return idx


def get_private_qa_dataset(version):
    return load_dataset(HIST_PRIVATE_QA_REPO_ID, revision=version)


def get_private_texts_dataset(version):
    return load_dataset(HIST_PRIVATE_TEXTS_REPO_ID, revision=version)


def build_private_to_public_text_mapping(version):
    """Map private_text_id -> public_text_id"""
    private_texts_ds = get_private_texts_dataset(version)
    mapping = {}
    for item in private_texts_ds["train"]:
        mapping[item["id"]] = item["public_id"]
    return mapping


def parse_text_ids(text_ids_str):
    """Parse text_ids string from private QA into flat list of int IDs.
    Handles nested lists like '[[1, 2], 3]' -> [1, 2, 3]
    """
    raw = ast.literal_eval(text_ids_str)
    flat = []
    for item in raw:
        if isinstance(item, list):
            flat.extend(int(x) for x in item)
        else:
            flat.append(int(item))
    return flat


def version_tuple(v):
    """Convert version string to tuple for comparison."""
    return tuple(int(x) for x in v.split("."))

In [8]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.random.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

## 2. Load LLM & Discover Versions

In [9]:
text_versions = set(get_ds_versions(HIST_TEXTS_REPO_ID))
question_versions = set(get_ds_versions(HIST_QUESTIONS_REPO_ID))
common_versions = text_versions & question_versions

# Filter to [MIN_VERSION, MAX_VERSION] range
min_t, max_t = version_tuple(MIN_VERSION), version_tuple(MAX_VERSION)
versions = sort_versions([
    v for v in common_versions
    if min_t <= version_tuple(v) <= max_t
])

print(f"Found {len(versions)} versions: {versions}")

Found 15 versions: ['1.0.0', '1.1.0', '1.2.0', '1.3.0', '1.4.0', '1.5.0', '1.6.0', '1.7.0', '1.8.0', '1.9.0', '1.10.0', '1.11.0', '1.12.0', '1.13.0', '1.15.0']


In [ ]:
%%time

# Kaggle FIX
os.environ["LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + os.environ.get("LIBRARY_PATH", "")
print("LIBRARY_PATH =", os.environ["LIBRARY_PATH"])

llm = VLLM(
    model=LLM_NAME,
    tensor_parallel_size=2,
    max_new_tokens=256,
    top_p=0.95,
    temperature=0.3,
    vllm_kwargs={
        "gpu_memory_utilization": 0.45,
        "max_num_batched_tokens": 8192,
        "max_model_len": 4096,
        "disable_log_stats": True,
        "seed": RANDOM_SEED
    },
    disable_log_stats=True,
)
tok = AutoTokenizer.from_pretrained(LLM_NAME)

# Отключаем внутренние tqdm-бары VLLM (Adding requests / Processed prompts)
_orig_generate = llm.client.generate
def _quiet_generate(*args, **kwargs):
    kwargs['use_tqdm'] = False
    return _orig_generate(*args, **kwargs)
llm.client.generate = _quiet_generate

In [11]:
# Шаблон промпта с контекстом (grounded)
messages_with_ctx = [
    {"role": "system", "content": LLM_PROMPT},
    {"role": "user", "content": "Заданный контекст:\n\n```text\n{context}\n```\n\nВопрос пользователя: {question}"},
]
template_with_ctx = tok.apply_chat_template(
    messages_with_ctx, tokenize=False, add_generation_prompt=True
)

# Шаблон промпта без контекста (ungrounded)
messages_no_ctx = [
    {"role": "system", "content": LLM_PROMPT_NO_CONTEXT},
    {"role": "user", "content": "Вопрос: {question}"},
]
template_no_ctx = tok.apply_chat_template(
    messages_no_ctx, tokenize=False, add_generation_prompt=True
)

print("Grounded template (preview):")
print(template_with_ctx, "...")
print("\nUngrounded template (preview):")
print(template_no_ctx, "...")

Grounded template (preview):
<|im_start|>system
Проанализируйте заданный контекст и ответьте на вопрос пользователя на основе сведений, предоставленных в этом контексте.
Не давайте никаких объяснений и пояснений к своему ответу. Не пишите ничего лишнего. Не извиняйтесь, не стройте диалог. Выдавайте только ответ и ничего больше.
Отвечайте на русском языке.
Если в заданном контексте нет информации для ответа на вопрос пользователя, то ничего не придумывайте и просто откажитесь отвечать.
<|im_end|>
<|im_start|>user
Заданный контекст:

```text
{context}
```

Вопрос пользователя: {question}<|im_end|>
<|im_start|>assistant
 ...

Ungrounded template (preview):
<|im_start|>system
Ответьте на вопрос пользователя.
Не давайте никаких объяснений. Выдавайте только ответ и ничего больше.
Отвечайте на русском языке.
<|im_end|>
<|im_start|>user
Вопрос: {question}<|im_end|>
<|im_start|>assistant
 ...


## 3. Generate Answers for All Versions

For each version (newest first):
1. Load public texts, public questions, private QA, and private-to-public text mapping
2. Skip questions already processed in a newer version
3. Get ground truth document IDs from private QA (`text_ids`) and map to public text IDs
4. Generate **grounded** (with ground truth context) and **ungrounded** (without context) answers
5. Collect dataset records

In [30]:
versions

['1.0.0',
 '1.1.0',
 '1.2.0',
 '1.3.0',
 '1.4.0',
 '1.5.0',
 '1.6.0',
 '1.7.0',
 '1.8.0',
 '1.9.0',
 '1.10.0',
 '1.11.0',
 '1.12.0',
 '1.13.0',
 '1.15.0']

In [35]:
%%time

from tqdm import tqdm

dataset_records = []

# Бюджет токенов для контекста
MAX_MODEL_LEN = 4096
MAX_NEW_TOKENS = 256

def truncate_context(context, question):
    """Обрезает контекст, чтобы промпт влезал в max_model_len."""
    full_prompt = template_with_ctx.replace("{context}", context).replace("{question}", question)
    prompt_tokens = tok.encode(full_prompt)
    budget = MAX_MODEL_LEN - MAX_NEW_TOKENS
    if len(prompt_tokens) <= budget:
        return context
    context_tokens = tok.encode(context)
    overflow = len(prompt_tokens) - budget
    trimmed = context_tokens[:len(context_tokens) - overflow]
    return tok.decode(trimmed, skip_special_tokens=True)

for ver in reversed(versions):
    print(f"\n{'='*60}")
    print(f"Processing version {ver}")
    print(f"{'='*60}")

    texts_ds_ver = load_dataset(HIST_TEXTS_REPO_ID, revision=ver)
    questions_ds_ver = load_dataset(HIST_QUESTIONS_REPO_ID, revision=ver)

    print(f"  Total questions: {len(questions_ds_ver['train'])}")

    try:
        qa_ds_ver = get_private_qa_dataset(ver)
    except Exception as e:
        print(f"  Warning: no private QA for version {ver}: {e}, skipping")
        continue

    priv_to_pub = build_private_to_public_text_mapping(ver)

    qa_idx = {}
    for item in qa_ds_ver["train"]:
        qa_idx[item["public_id"]] = {
            "answer": item["answer"],
            "text_ids": item["text_ids"],
        }

    t_idx = _build_text_index(texts_ds_ver)

    all_items = list(questions_ds_ver["train"])
    skipped = 0
    truncated_cnt = 0

    for item in tqdm(all_items, desc=f"v{ver}", mininterval=1.0):
        qid = item["id"]
        unique_qid = f"{ver}_{qid}"
        question = item["question"]
        qa_info = qa_idx.get(qid, {})
        ref_answer = qa_info.get("answer", "")
        text_ids_str = qa_info.get("text_ids", "[]")

        private_text_ids = parse_text_ids(text_ids_str)
        public_text_ids = []
        for priv_id in private_text_ids:
            pub_id = priv_to_pub.get(priv_id)
            if pub_id is not None:
                public_text_ids.append(pub_id)

        evidence_texts = [t_idx.get(pid, "") for pid in public_text_ids]

        if not evidence_texts or all(t == "" for t in evidence_texts):
            skipped += 1
            continue

        # ── Grounded answer ──
        context = "\n\n".join(evidence_texts)
        context_truncated = truncate_context(context, question)
        if len(context_truncated) < len(context):
            truncated_cnt += 1

        prompt_grounded = template_with_ctx.replace("{context}", context_truncated).replace("{question}", question)
        answer_grounded = llm.invoke(prompt_grounded)

        dataset_records.append({
            "question_id": unique_qid,
            "question": question,
            "reference_answer": ref_answer,
            "model_answer": answer_grounded,
            "found_ids": public_text_ids,
            "evidence_texts": evidence_texts,
            "is_grounded": True,
        })

        # ── Ungrounded answer ──
        prompt_ungrounded = template_no_ctx.replace("{question}", question)
        answer_ungrounded = llm.invoke(prompt_ungrounded)

        dataset_records.append({
            "question_id": unique_qid,
            "question": question,
            "reference_answer": ref_answer,
            "model_answer": answer_ungrounded,
            "found_ids": public_text_ids,
            "evidence_texts": evidence_texts,
            "is_grounded": False,
        })

    if skipped:
        print(f"  Skipped {skipped} questions (no evidence texts)")
    if truncated_cnt:
        print(f"  Truncated context: {truncated_cnt} questions")
    print(f"  Done. Records so far: {len(dataset_records)}")

n_grounded = sum(1 for r in dataset_records if r["is_grounded"])
n_ungrounded = sum(1 for r in dataset_records if not r["is_grounded"])
print(f"\n{'='*60}")
print(f"Total: {len(dataset_records)} examples (grounded: {n_grounded}, ungrounded: {n_ungrounded})")
print(f"Unique questions: {len(set(r['question_id'] for r in dataset_records))}")


Processing version 1.15.0
  Total questions: 600


v1.15.0: 100%|██████████| 600/600 [04:32<00:00,  2.20it/s]


  Truncated context: 13 questions
  Done. Records so far: 1200

Processing version 1.13.0
  Total questions: 600


v1.13.0: 100%|██████████| 600/600 [04:33<00:00,  2.19it/s]


  Truncated context: 6 questions
  Done. Records so far: 2400

Processing version 1.12.0
  Total questions: 600


...

  Truncated context: 13 questions
  Done. Records so far: 15600

Processing version 1.1.0
  Total questions: 600


v1.1.0: 100%|██████████| 600/600 [04:29<00:00,  2.23it/s]


  Truncated context: 4 questions
  Done. Records so far: 16800

Processing version 1.0.0
  Total questions: 600


v1.0.0: 100%|██████████| 600/600 [04:18<00:00,  2.32it/s]

  Truncated context: 4 questions
  Done. Records so far: 18000

Total: 18000 examples (grounded: 9000, ungrounded: 9000)
Unique questions: 9000
CPU times: user 2min 51s, sys: 17.5 s, total: 3min 9s
Wall time: 1h 9min


## 4. Save Full Dataset

In [36]:
len(dataset_records)

18000

In [37]:
dataset_records[0]

{'question_id': '1.15.0_0',
 'question': 'Какой регион был определён в качестве приоритетного для Дональда Трампа?',
 'reference_answer': 'Индо-Тихоокеанский регион',
 'model_answer': 'Индхо-Тихоокеанский',
 'found_ids': [197],
 'evidence_texts': ['Бывший советник офиса президента Украины Алексей Арестович в своем Telegram-канале заявил, что президент США Дональд Трамп хочет закончить конфликт на Украине в ближайшие дни, чтобы начать военную кампанию против Венесуэлы. \nПо его словам, перемирие на Украине является лишь частью большой стратегии американского лидера в рамках глобального противостояния с Китаем. На данный момент, по словам политика, глава Белого дома подготавливает почву, чтобы США выиграли гонку на длинной дистанции у Китая и России.  \nДля реализации задуманного, по мнению Арестовича, Трампу необходимо навести порядок в Западном полушарии в соответствии с доктриной Монро и перенести основные усилия в Индо-Тихоокеанский регион.  \n"В планах Трампа - закончить войну в Укр

In [38]:
results.save(dataset_records, "./fact_check_eval_dataset.json")
print(f"Saved fact-checking dataset: {len(dataset_records)} examples")
print(f"  Grounded: {sum(1 for r in dataset_records if r['is_grounded'])}")
print(f"  Ungrounded: {sum(1 for r in dataset_records if not r['is_grounded'])}")

Saved fact-checking dataset: 18000 examples
  Grounded: 9000
  Ungrounded: 9000


## 5. Train/Val/Test Split

Split by `question_id` so that both grounded and ungrounded examples for the same question stay in the same split. Proportions: 70% train / 15% val / 15% test.

In [39]:
# Train/val/test split by question_id
# Same question's grounded + ungrounded examples stay in the same split
unique_qids = list(set(r["question_id"] for r in dataset_records))
random.shuffle(unique_qids)

n_total = len(unique_qids)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

train_qids = set(unique_qids[:n_train])
val_qids = set(unique_qids[n_train:n_train + n_val])
test_qids = set(unique_qids[n_train + n_val:])

train_records = [r for r in dataset_records if r["question_id"] in train_qids]
val_records = [r for r in dataset_records if r["question_id"] in val_qids]
test_records = [r for r in dataset_records if r["question_id"] in test_qids]

print(f"Train: {len(train_records)} examples ({len(train_qids)} questions)")
print(f"Val:   {len(val_records)} examples ({len(val_qids)} questions)")
print(f"Test:  {len(test_records)} examples ({len(test_qids)} questions)")
print(f"Total: {len(train_records) + len(val_records) + len(test_records)}")

Train: 12600 examples (6300 questions)
Val:   2700 examples (1350 questions)
Test:  2700 examples (1350 questions)
Total: 18000


In [40]:
# Save splits locally
import os as _os

save_dir = "rag_fact_checking/DEREC/dataset/DRAGON"
_os.makedirs(save_dir, exist_ok=True)

results.save(train_records, f"{save_dir}/train.json")
results.save(val_records, f"{save_dir}/val.json")
results.save(test_records, f"{save_dir}/test.json")

print(f"Saved to {save_dir}/")
print(f"  train.json: {len(train_records)} examples")
print(f"  val.json:   {len(val_records)} examples")
print(f"  test.json:  {len(test_records)} examples")

Saved to rag_fact_checking/DEREC/dataset/DRAGON/
  train.json: 12600 examples
  val.json:   2700 examples
  test.json:  2700 examples


## 6. Publish Dataset to HuggingFace

Upload train/val/test splits to HuggingFace Hub so that `train_dragon.ipynb` can load them via `load_dataset()`.

In [41]:
ds_dict = DatasetDict({
    "train": Dataset.from_list(train_records),
    "val": Dataset.from_list(val_records),
    "test": Dataset.from_list(test_records),
})

ds_dict.push_to_hub(HF_DATASET_REPO, token=HF_TOKEN)
print(f"Published to https://huggingface.co/datasets/{HF_DATASET_REPO}")
print(f"  train: {len(train_records)}, val: {len(val_records)}, test: {len(test_records)}")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/13 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Published to https://huggingface.co/datasets/Makson4ic/dragon-derec-dataset
  train: 12600, val: 2700, test: 2700
